# FactGenius v1: Evaluate BERT đã fine-tune trên `llm_v1/test.csv`

Notebook này chỉ dùng bộ dữ liệu `llm_v1`, cụ thể là file:

```text
llm_v1/test.csv
```

Mục tiêu:

1. Load model BERT đã fine-tune sẵn từ Hugging Face.
2. Chạy dự đoán trên tập test của `llm_v1`.
3. In accuracy/F1/classification report tổng thể.
4. In bảng accuracy gộp theo các relation xuất hiện trong evidence.
5. Nếu có raw metadata test gốc, có thể gộp thêm theo reasoning type như `multi hop`, `existence`, `multi claim`, `negation`, `num1`.

Lưu ý quan trọng: `llm_v1/test.csv` hiện chỉ có `Sentence,Label`, không còn cột metadata reasoning type gốc. Vì vậy notebook mặc định gộp theo **relation trong evidence**, ví dụ `successor`, `parentCompany`, `birthPlace`. Phần gộp theo `multi-hop/conjunction/existence/...` cần file metadata gốc nếu bạn có.

## 1. Cài thư viện và kiểm tra GPU

In [ ]:
%pip -q install -U pandas numpy scikit-learn transformers datasets accelerate tqdm

In [ ]:
import os
import re
import ast
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

RANDOM_SEED = 2024
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Chọn nơi đặt `llm_v1/test.csv`

Bạn chỉ cần đưa folder `llm_v1` lên Colab hoặc Drive. Không cần clone repo.

Ví dụ nếu dùng `/content`:

```text
/content/llm_v1/test.csv
```

Ví dụ nếu dùng Google Drive:

```text
/content/drive/MyDrive/FactGenius/llm_v1/test.csv
```

In [ ]:
DATA_STORAGE = 'content'  # @param ['content', 'drive']

# Root chứa folder llm_v1 nếu dùng /content.
CONTENT_DATA_ROOT = '/content'  # @param {type:'string'}

# Root chứa folder llm_v1 nếu dùng Drive.
DRIVE_DATA_ROOT = '/content/drive/MyDrive/FactGenius'  # @param {type:'string'}

# Folder dữ liệu v1. Notebook này chỉ dùng v1.
DATA_FOLDER_NAME = 'llm_v1'

# Nơi lưu kết quả CSV sau khi evaluate.
OUTPUT_DIR = '/content/factgenius_v1_eval_outputs'  # @param {type:'string'}

if DATA_STORAGE == 'drive':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ModuleNotFoundError:
        print('Không chạy trong Colab, bỏ qua mount Drive.')

data_root = Path(DRIVE_DATA_ROOT if DATA_STORAGE == 'drive' else CONTENT_DATA_ROOT)
DATA_DIR = data_root / DATA_FOLDER_NAME
TEST_CSV_PATH = DATA_DIR / 'test.csv'
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('TEST_CSV_PATH =', TEST_CSV_PATH)
print('OUTPUT_DIR    =', OUTPUT_DIR)

### 2.1. Tùy chọn upload zip chứa `llm_v1`

Nếu chưa có folder `llm_v1` trong `/content`, bạn có thể zip folder `llm_v1`, upload, rồi giải nén bằng cell này.

Cấu trúc zip nên là:

```text
llm_v1/test.csv
llm_v1/train.csv
llm_v1/val.csv
```

Notebook này chỉ cần `test.csv`, nhưng giữ nguyên folder sẽ tiện hơn.

In [ ]:
UPLOAD_AND_UNZIP_DATA = False  # @param {type:'boolean'}

if UPLOAD_AND_UNZIP_DATA:
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename in uploaded.keys():
            archive_path = Path('/content') / filename
            print('Giải nén:', archive_path)
            shutil.unpack_archive(str(archive_path), str(data_root))
        print('Giải nén xong vào:', data_root)
    except ModuleNotFoundError:
        print('Cell upload chỉ chạy trong Google Colab.')
else:
    print('Bỏ qua upload/unzip.')

## 3. Load `llm_v1/test.csv` và kiểm tra dữ liệu

In [ ]:
if not TEST_CSV_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy {TEST_CSV_PATH}. Hãy upload folder llm_v1 hoặc sửa DATA path.')

test_df = pd.read_csv(TEST_CSV_PATH)

required_columns = {'Sentence', 'Label'}
missing = required_columns - set(test_df.columns)
if missing:
    raise ValueError(f'test.csv thiếu cột: {missing}')

def normalize_label(value):
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return 1
    if text in {'false', '0', 'no'}:
        return 0
    raise ValueError(f'Label không hợp lệ: {value!r}')

test_df['labels'] = test_df['Label'].map(normalize_label).astype(int)

print('Số dòng test:', len(test_df))
print(test_df['labels'].value_counts().rename(index={0: 'False', 1: 'True'}))
display(test_df.head(5))

## 4. Parse claim, evidence và relation trong evidence

`llm_v1/test.csv` có cột `Sentence` dạng:

```text
Claim: ... Evidence: Entity_A >- relation -> Entity_B | Entity_C >- relation2 -> Entity_D
```

Cell này tách ra:

- `claim_text`
- `evidence_text`
- `evidence_relations`: danh sách relation trong evidence
- `primary_relation`: relation đầu tiên, dùng để group một dòng vào một relation chính
- `num_evidence_paths`: số path evidence trong dòng

In [ ]:
def split_claim_evidence(sentence):
    text = str(sentence)
    if ' Evidence: ' in text:
        claim_part, evidence_part = text.split(' Evidence: ', 1)
        claim = claim_part.replace('Claim: ', '').strip()
        evidence = evidence_part.strip()
    else:
        claim = text.replace('Claim: ', '').strip()
        evidence = ''
    return claim, evidence

def extract_relations_from_evidence(evidence):
    evidence = str(evidence)
    if not evidence.strip():
        return []
    relations = []
    for part in evidence.split('|'):
        part = part.strip()
        match = re.search(r'>-\s*(.*?)\s*->', part)
        if match:
            relation = match.group(1).strip()
            if relation.startswith('~'):
                relation = relation[1:]
            if relation:
                relations.append(relation)
    return relations

parsed = test_df['Sentence'].map(split_claim_evidence)
test_df['claim_text'] = parsed.map(lambda x: x[0])
test_df['evidence_text'] = parsed.map(lambda x: x[1])
test_df['evidence_relations'] = test_df['evidence_text'].map(extract_relations_from_evidence)
test_df['primary_relation'] = test_df['evidence_relations'].map(lambda rels: rels[0] if rels else 'NO_EVIDENCE')
test_df['num_evidence_paths'] = test_df['evidence_text'].map(lambda x: 0 if not str(x).strip() else len([p for p in str(x).split('|') if p.strip()]))

display(test_df[['claim_text', 'evidence_text', 'evidence_relations', 'primary_relation', 'num_evidence_paths', 'Label']].head(10))
print('Top primary relations:')
display(test_df['primary_relation'].value_counts().head(20).to_frame('count'))

## 5. Load BERT đã fine-tune trên `llm_v1`

Model BERT được README gốc nhắc tới cho dữ liệu `llm_v1`:

```text
SushantGautam/KG-LLM-bert-base
```

Notebook mặc định dùng model này. Nếu muốn so sánh với RoBERTa two-stage, có thể đổi thành:

```text
SushantGautam/KG-LLM-roberta-base
```

In [ ]:
HF_MODEL_ID = 'SushantGautam/KG-LLM-bert-base'  # @param {type:'string'}
MAX_LENGTH = 512  # @param {type:'integer'}
BATCH_SIZE = 32  # @param {type:'integer'}

# Nếu muốn chạy nhanh để smoke test, bật True và giảm TEST_SAMPLE_SIZE.
QUICK_RUN = False  # @param {type:'boolean'}
TEST_SAMPLE_SIZE = 1000  # @param {type:'integer'}

eval_df = test_df.copy().reset_index(drop=False).rename(columns={'index': 'original_row_id'})
if QUICK_RUN and TEST_SAMPLE_SIZE < len(eval_df):
    eval_df = eval_df.sample(n=TEST_SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

print('Model:', HF_MODEL_ID)
print('Số dòng evaluate:', len(eval_df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_ID)

eval_dataset = Dataset.from_pandas(eval_df[['Sentence', 'labels']], preserve_index=False)

def tokenize_batch(batch):
    return tokenizer(batch['Sentence'], truncation=True, max_length=MAX_LENGTH)

eval_dataset = eval_dataset.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

eval_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'tmp_eval'),
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=eval_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Load model và dataset xong.')

## 6. Chạy evaluate trên test set

In [ ]:
prediction_output = trainer.predict(eval_dataset)

logits = prediction_output.predictions
y_true = prediction_output.label_ids
y_pred = np.argmax(logits, axis=1)
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

print('Metrics từ Trainer:')
print(prediction_output.metrics)

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=['False', 'True'], zero_division=0))

print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))

results_df = eval_df.copy()
results_df['predicted_label'] = y_pred
results_df['gold_text'] = results_df['labels'].map({0: 'False', 1: 'True'})
results_df['predicted_text'] = results_df['predicted_label'].map({0: 'False', 1: 'True'})
results_df['prob_false'] = probs[:, 0]
results_df['prob_true'] = probs[:, 1]
results_df['correct'] = (results_df['labels'] == results_df['predicted_label']).astype(int)

display(results_df[['original_row_id', 'Sentence', 'gold_text', 'predicted_text', 'prob_false', 'prob_true', 'correct']].head(10))

## 7. Bảng độ chính xác gộp theo relation trong evidence

Có 2 bảng:

1. Gộp theo `primary_relation`: mỗi dòng test chỉ thuộc relation đầu tiên trong evidence.
2. Gộp theo `all_evidence_relations`: một dòng test có nhiều relation sẽ được tính vào nhiều group. Bảng này giúp xem một relation xuất hiện ở đâu thì model đúng/sai thế nào.

Mặc định chỉ hiển thị relation có số mẫu tối thiểu `MIN_GROUP_SIZE` để bảng không bị nhiễu bởi group quá nhỏ.

In [ ]:
MIN_GROUP_SIZE = 20  # @param {type:'integer'}

def make_group_table(df, group_col, min_size=1):
    grouped = (
        df.groupby(group_col)
        .agg(
            samples=('correct', 'size'),
            accuracy=('correct', 'mean'),
            true_rate=('labels', 'mean'),
            pred_true_rate=('predicted_label', 'mean'),
            avg_prob_true=('prob_true', 'mean'),
        )
        .reset_index()
    )
    grouped = grouped[grouped['samples'] >= min_size].copy()
    grouped['accuracy'] = grouped['accuracy'].round(4)
    grouped['true_rate'] = grouped['true_rate'].round(4)
    grouped['pred_true_rate'] = grouped['pred_true_rate'].round(4)
    grouped['avg_prob_true'] = grouped['avg_prob_true'].round(4)
    return grouped.sort_values(['accuracy', 'samples'], ascending=[False, False])

primary_relation_table = make_group_table(results_df, 'primary_relation', MIN_GROUP_SIZE)
print('Accuracy theo primary_relation:')
display(primary_relation_table)

primary_relation_path = OUTPUT_DIR / 'bert_v1_accuracy_by_primary_relation.csv'
primary_relation_table.to_csv(primary_relation_path, index=False)
print('Đã lưu:', primary_relation_path)

In [ ]:
exploded_relation_df = results_df.copy()
exploded_relation_df['relation_for_group'] = exploded_relation_df['evidence_relations'].map(lambda rels: rels if rels else ['NO_EVIDENCE'])
exploded_relation_df = exploded_relation_df.explode('relation_for_group')

all_relation_table = make_group_table(exploded_relation_df, 'relation_for_group', MIN_GROUP_SIZE)
print('Accuracy theo tất cả relation xuất hiện trong evidence:')
display(all_relation_table)

all_relation_path = OUTPUT_DIR / 'bert_v1_accuracy_by_all_evidence_relations.csv'
all_relation_table.to_csv(all_relation_path, index=False)
print('Đã lưu:', all_relation_path)

## 8. Bảng độ chính xác theo độ phức tạp evidence

Vì `llm_v1/test.csv` không có metadata reasoning type, ta có thể xem thêm một proxy đơn giản: số path evidence trong input.

Ví dụ:

- `0`: không tìm được evidence.
- `1`: một evidence path.
- `2+`: nhiều evidence path.

In [ ]:
def evidence_bucket(n):
    if n == 0:
        return '0_path'
    if n == 1:
        return '1_path'
    if n == 2:
        return '2_paths'
    return '3plus_paths'

results_df['evidence_path_bucket'] = results_df['num_evidence_paths'].map(evidence_bucket)
path_bucket_table = make_group_table(results_df, 'evidence_path_bucket', 1)
print('Accuracy theo số evidence paths:')
display(path_bucket_table)

path_bucket_path = OUTPUT_DIR / 'bert_v1_accuracy_by_evidence_path_count.csv'
path_bucket_table.to_csv(path_bucket_path, index=False)
print('Đã lưu:', path_bucket_path)

## 9. Optional: gộp theo reasoning type gốc nếu có metadata

Repo hiện tại không có raw `test.csv` gốc chứa metadata. Nếu bạn có file raw test gốc với cột `Metadata` hoặc `Metatada`, điền path ở dưới.

Các type thường gặp trong code gốc:

```text
negation
num1
multi claim
existence
multi hop
```

Cell này sẽ merge theo thứ tự dòng. Vì vậy file metadata phải cùng thứ tự với `llm_v1/test.csv`.

In [ ]:
RAW_TEST_METADATA_CSV = ''  # @param {type:'string'}

def parse_metadata_cell(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text = str(value)
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            return [parsed]
        return list(parsed) if hasattr(parsed, '__iter__') else [str(parsed)]
    except Exception:
        return [text]

def choose_reasoning_type(metadata_items):
    items = set(str(x) for x in metadata_items)
    if 'negation' in items:
        return 'negation'
    for key in ['num1', 'multi claim', 'existence', 'multi hop']:
        if key in items:
            return key
    return 'other'

if RAW_TEST_METADATA_CSV.strip():
    raw_meta_path = Path(RAW_TEST_METADATA_CSV.strip())
    if not raw_meta_path.exists():
        raise FileNotFoundError(raw_meta_path)

    raw_test_df = pd.read_csv(raw_meta_path)
    meta_col = 'Metadata' if 'Metadata' in raw_test_df.columns else ('Metatada' if 'Metatada' in raw_test_df.columns else None)
    if meta_col is None:
        raise ValueError('Không tìm thấy cột Metadata hoặc Metatada trong raw test CSV.')

    if len(raw_test_df) != len(test_df):
        print('Cảnh báo: số dòng metadata khác số dòng llm_v1/test.csv. Sẽ merge theo số dòng nhỏ hơn.')

    reasoning_df = results_df.copy()
    metadata_series = raw_test_df[meta_col].iloc[reasoning_df['original_row_id'].to_numpy()].reset_index(drop=True)
    reasoning_df['metadata_items'] = metadata_series.map(parse_metadata_cell)
    reasoning_df['reasoning_type'] = reasoning_df['metadata_items'].map(choose_reasoning_type)

    reasoning_table = make_group_table(reasoning_df, 'reasoning_type', 1)
    print('Accuracy theo reasoning type gốc:')
    display(reasoning_table)

    reasoning_path = OUTPUT_DIR / 'bert_v1_accuracy_by_reasoning_type.csv'
    reasoning_table.to_csv(reasoning_path, index=False)
    print('Đã lưu:', reasoning_path)
else:
    print('Chưa cấu hình RAW_TEST_METADATA_CSV, bỏ qua bảng reasoning type gốc.')

## 10. Lưu toàn bộ prediction

In [ ]:
predictions_path = OUTPUT_DIR / 'bert_v1_test_predictions.csv'
results_df.to_csv(predictions_path, index=False)
print('Đã lưu toàn bộ prediction:', predictions_path)

print('\nCác file output:')
for path in sorted(OUTPUT_DIR.glob('*.csv')):
    print('-', path)